# EduVision_DV — 01. Data Loading & Validation

In [2]:
import pandas as pd
import os

RAW_DIR = os.path.join(".", "..", "data", "raw")
DOCS_DIR = os.path.join(".", "..", "docs")
os.makedirs(DOCS_DIR, exist_ok=True)

DATASETS = [
    {
        "name": "QS World University Rankings 2025",
        "source": "QS Quacquarelli Symonds (Kaggle mirror)",
        "url": "https://www.kaggle.com/datasets/joebeachcapital/qs-world-university-rankings-2025",
        "year": 2025,
        "file": "qs_2025_raw.csv",
        "encoding": "latin1",
        "uni_col": "Institution_Name",
        "country_col": "Location",
        "indicators": ["Academic_Reputation_Score", "Employer_Reputation_Score",
                       "Faculty_Student_Score", "Citations_per_Faculty_Score",
                       "International_Students_Score", "Overall_Score"],
    },
    {
        "name": "Times Higher Education World University Rankings 2024",
        "source": "Times Higher Education (Kaggle mirror)",
        "url": "https://www.kaggle.com/datasets/thedevastator/2024-university-world-rankings",
        "year": 2024,
        "file": "the_2024_raw.csv",
        "encoding": "utf-8",
        "uni_col": "name",
        "country_col": "location",
        "indicators": ["scores_overall", "scores_teaching", "scores_research",
                       "scores_citations", "stats_student_staff_ratio", "stats_pc_intl_students"],
    },
    {
        "name": "World University Rankings 2023",
        "source": "Times Higher Education, via alitaqi000 (Kaggle mirror)",
        "url": "https://www.kaggle.com/datasets/alitaqi000/world-university-rankings-2023",
        "year": 2023,
        "file": "wur_2023_raw.csv",
        "encoding": "utf-8",
        "uni_col": "Name of University",
        "country_col": "Location",
        "indicators": ["OverAll Score", "Teaching Score", "Research Score",
                   "Citations Score", "No of student per staff", "International Student"],
        "expected_rows": 1799,
    },
]

WORLD_BANK_FILE = "world_bank_education_subset.csv"
WORLD_BANK_SPEC = {
    "name": "World Bank Education Statistics",
    "source": "World Bank EdStats (Kaggle: theworldbank/education-statistics)",
    "url": "https://www.kaggle.com/datasets/theworldbank/education-statistics",
    "year": "multi-year panel (1970-2020s)",
    "file": WORLD_BANK_FILE,
    "encoding": "utf-8",
    "uni_col": None,          # country-level dataset, no university column
    "country_col": "Country Name",
    "indicators": ["SE.XPD.TOTL.GD.ZS", "SE.TER.ENRR", "SE.ADT.LITR.ZS",
                   "SE.XPD.TOTL.GB.ZS", "SE.PRM.ENRR"],
}


def validate_country_dataset(spec):
    """World Bank EdStats is shaped differently (long panel, one row per
    country+indicator, one column per year) - validate it on its own terms
    rather than forcing it through the university-shaped checks above."""
    df = pd.read_csv(os.path.join(RAW_DIR, spec["file"]), encoding=spec["encoding"])
    year_cols = [c for c in df.columns if c.strip().isdigit()]
    indicator_rows = df[df["Indicator Code"].isin(spec["indicators"])]

    lines = [
        f"### {spec['name']}",
        f"- **Source:** {spec['source']}",
        f"- **URL:** {spec['url']}",
        f"- **Year:** {spec['year']}",
        f"- **Rows (full file):** {len(df)}",
        f"- **Columns:** {len(df.columns)} ({len(year_cols)} year columns)",
        f"- **University Name Column:** n/a (country-level dataset)",
        f"- **Country Column:** `{spec['country_col']}`",
        f"- **Important Indicators (of {df['Indicator Code'].nunique()} total in file):** {', '.join(spec['indicators'])}",
        f"- **Rows matching our 5 selected indicators:** {len(indicator_rows)}",
        f"- **Countries covered by selected indicators:** {indicator_rows['Country Name'].nunique()}",
        f"- **Duplicate rows:** {df.duplicated().sum()}",
        "",
    ]
    return "\n".join(lines)


def validate(spec):
    df = pd.read_csv(
        os.path.join(RAW_DIR, spec["file"]),
        encoding=spec["encoding"]
    )

    missing_pct = (
        df.isna().mean() * 100
    ).round(1).sort_values(ascending=False)

    missing_top = missing_pct[missing_pct > 0].head(8)

    dup_rows = df.duplicated().sum()
    dup_unis = df[spec["uni_col"]].duplicated().sum()

    actual_rows = len(df)

    lines = [
        f"### {spec['name']}",
        f"- **Source:** {spec['source']}",
        f"- **URL:** {spec['url']}",
        f"- **Year:** {spec['year']}",
        f"- **Rows:** {actual_rows}",
        f"- **Columns:** {len(df.columns)}",
        f"- **University Name Column:** `{spec['uni_col']}`",
        f"- **Country Column:** `{spec['country_col']}`",
        f"- **Important Indicators:** {', '.join(spec['indicators'])}",
        f"- **Duplicate rows:** {dup_rows}",
        f"- **Duplicate university names:** {dup_unis}",
    ]

    if "expected_rows" in spec:
        expected = spec["expected_rows"]

        if actual_rows == expected:
            lines.append(
                f"- **Reference row-count check:** PASS ({actual_rows} rows)"
            )
        else:
            lines.append(
                f"- **Reference row-count check:** REVIEW "
                f"(reference: {expected} rows; actual: {actual_rows} rows)"
            )

    lines.append("- **Top missing-value columns (%):**")

    for col, pct in missing_top.items():
        lines.append(f"  - `{col}`: {pct}%")

    lines.append(
        f"- **Data types:** {dict(df.dtypes.astype(str).value_counts())}"
    )

    lines.append("")

    return "\n".join(lines), df

def main():
    report = ["# EduVision_DV - Dataset Validation Report",
               "Generated before any cleaning step, per project reference guide Section 8.\n"]
    for spec in DATASETS:
        section, df = validate(spec)
        report.append(section)
        print(section)
        print("-" * 60)

    wb_path = os.path.join(RAW_DIR, WORLD_BANK_FILE)
    if os.path.exists(wb_path):
        section = validate_country_dataset(WORLD_BANK_SPEC)
        report.append(section)
        print(section)
    else:
        note = (f"### {WORLD_BANK_SPEC['name']}\n"
                f"- **Status:** NOT YET PROVIDED - expected at `data/raw/{WORLD_BANK_FILE}`\n"
                f"- Download from {WORLD_BANK_SPEC['url']}, save it there, and re-run this notebook.\n")
        report.append(note)
        print(note)

    out = os.path.join(DOCS_DIR, "01_dataset_validation_report.md")
    with open(out, "w") as f:
        f.write("\n".join(report))
    print(f"\nSaved -> {out}")


if __name__ == "__main__":
    main()


### QS World University Rankings 2025
- **Source:** QS Quacquarelli Symonds (Kaggle mirror)
- **URL:** https://www.kaggle.com/datasets/joebeachcapital/qs-world-university-rankings-2025
- **Year:** 2025
- **Rows:** 1503
- **Columns:** 28
- **University Name Column:** `Institution_Name`
- **Country Column:** `Location`
- **Important Indicators:** Academic_Reputation_Score, Employer_Reputation_Score, Faculty_Student_Score, Citations_per_Faculty_Score, International_Students_Score, Overall_Score
- **Duplicate rows:** 0
- **Duplicate university names:** 0
- **Top missing-value columns (%):**
  - `Overall_Score`: 60.0%
  - `International_Faculty_Rank`: 6.7%
  - `International_Faculty_Score`: 6.7%
  - `International_Students_Score`: 3.9%
  - `International_Students_Rank`: 3.9%
  - `STATUS`: 2.5%
  - `RANK_2024`: 1.4%
  - `Sustainability_Score`: 1.3%
- **Data types:** {'object': np.int64(19), 'float64': np.int64(9)}

------------------------------------------------------------
### Times Higher